# EDA — Legacy SMART Outcome Columns

Goal: understand the structure of `edoodvas`, `ebero_n`, `emi_n` (and their sibling columns)
to decide how to handle indicator value 2 (lost to follow-up) in `compute_legacy_targets`.

Key question: for status-2 patients, does the corresponding `e*_f` column already contain
a meaningful censoring time, or is an external `censoring_time` parameter required?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# ── adjust this path ──────────────────────────────────────────────────────────
SMART_CSV = "data/dummy_data/smart/smart.csv"
# ─────────────────────────────────────────────────────────────────────────────

df = pd.read_csv(SMART_CSV)
print(f"Shape: {df.shape}")
df[["edood_f", "edood_n", "edoodvas",
    "ebero_f", "ebero_n", "ebero_s",
    "emi_f",   "emi_n",   "emi_s"]].head()

## 1. Indicator value distributions

In [ ]:
indicators = {"death (edoodvas)": "edoodvas", "stroke (ebero_n)": "ebero_n", "MI (emi_n)": "emi_n"}

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (label, col) in zip(axes, indicators.items()):
    counts = df[col].value_counts().sort_index()
    ax.bar(counts.index.astype(str), counts.values)
    ax.set_title(label)
    ax.set_xlabel("indicator value  (0=censored, 1=event, 2=lost-to-FU)")
    ax.set_ylabel("patients")
    for i, v in enumerate(counts.values):
        ax.text(i, v + counts.values.max() * 0.01, str(v), ha="center", fontsize=9)
plt.tight_layout()
plt.show()

for label, col in indicators.items():
    print(f"\n{label}:")
    print(df[col].value_counts().sort_index().rename("count").to_frame())

## 2. Cross-tabulation of indicator combinations

Checks the assertion: if one indicator is 2, are all three 2?

In [ ]:
combo = df[["edoodvas", "ebero_n", "emi_n"]].astype(int)
combo_counts = combo.value_counts().reset_index().rename(columns={0: "count"})
combo_counts.columns = ["edoodvas", "ebero_n", "emi_n", "count"]
combo_counts = combo_counts.sort_values("count", ascending=False)
print(f"Distinct indicator combinations: {len(combo_counts)}")
combo_counts

In [ ]:
# Flag rows where LTFU is not all-or-nothing
has_any_ltfu = (combo == 2).any(axis=1)
all_ltfu     = (combo == 2).all(axis=1)
partial_ltfu = has_any_ltfu & ~all_ltfu
print(f"Patients with any indicator==2:  {has_any_ltfu.sum()}")
print(f"Patients with all indicators==2: {all_ltfu.sum()}")
print(f"Patients with partial LTFU:      {partial_ltfu.sum()}  ← should be 0")
if partial_ltfu.any():
    display(df.loc[partial_ltfu, ["m3life_no", "edoodvas", "ebero_n", "emi_n"]].head(20))

## 3. Are `e*_f` times equal across endpoints for status-0 patients?

Checks the assertion: when all indicators are 0, all three follow-up times must agree.

In [ ]:
all_censored = (combo == 0).all(axis=1)
times_censored = df.loc[all_censored, ["edood_f", "ebero_f", "emi_f"]]
unequal = times_censored.nunique(axis=1) > 1

print(f"All-censored patients:            {all_censored.sum()}")
print(f"  with differing e*_f times:      {unequal.sum()}  ← should be 0")
if unequal.any():
    display(times_censored[unequal].head(20))

## 4. Key question: for status-2 patients, what do `e*_f` times look like?

If `e*_f` for lost-to-FU patients already holds a sensible censoring time (e.g. same as
the administrative cutoff, or all three times are equal), then an external `censoring_time`
parameter is not needed.

In [ ]:
all_ltfu_mask = (combo == 2).all(axis=1)
times_ltfu = df.loc[all_ltfu_mask, ["edood_f", "ebero_f", "emi_f"]]

print(f"All-LTFU patients: {all_ltfu_mask.sum()}")
if all_ltfu_mask.sum() > 0:
    unequal_ltfu = times_ltfu.nunique(axis=1) > 1
    print(f"  with differing e*_f times: {unequal_ltfu.sum()}")
    print(f"\nedood_f stats:")
    print(times_ltfu["edood_f"].describe())
    print(f"\nebero_f stats:")
    print(times_ltfu["ebero_f"].describe())
    print(f"\nemi_f stats:")
    print(times_ltfu["emi_f"].describe())

    fig, ax = plt.subplots(figsize=(8, 4))
    for col, label in zip(["edood_f", "ebero_f", "emi_f"], ["death", "stroke", "MI"]):
        ax.hist(times_ltfu[col].dropna(), bins=40, alpha=0.5, label=label)
    ax.set_title("e*_f distribution for all-LTFU patients (indicator==2)")
    ax.set_xlabel("days")
    ax.legend()
    plt.tight_layout()
    plt.show()

    if unequal_ltfu.any():
        print("\nSample of patients with differing e*_f times:")
        display(times_ltfu[unequal_ltfu].head(20))

## 5. Compare `e*_f` distributions across indicator values

Overlays the follow-up time distributions for status 0, 1, and 2 to see whether
status-2 times cluster near a common administrative cutoff or spread across the range.

In [ ]:
endpoint_pairs = [
    ("edoodvas", "edood_f", "Death"),
    ("ebero_n",  "ebero_f", "Stroke"),
    ("emi_n",    "emi_f",   "MI"),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (ind_col, time_col, title) in zip(axes, endpoint_pairs):
    for val, label, color in [(0, "censored (0)", "steelblue"),
                               (1, "event (1)",   "tomato"),
                               (2, "lost-to-FU (2)", "orange")]:
        mask = df[ind_col] == val
        if mask.sum() > 0:
            ax.hist(df.loc[mask, time_col].dropna(), bins=40,
                    alpha=0.55, label=f"{label} (n={mask.sum()})", color=color)
    ax.set_title(title)
    ax.set_xlabel("days")
    ax.legend(fontsize=8)
plt.suptitle("Follow-up time (e*_f) by indicator value", y=1.02)
plt.tight_layout()
plt.show()

## 6. Event subtype distributions (stroke and MI)

Only events with indicator==1 are relevant here.
Checks which `ebero_s` / `emi_s` values are present and whether any fall outside
the expected subtypes `[11, 102]` / `[41, 101]`.

In [ ]:
for ind_col, subtype_col, valid, title in [
    ("ebero_n", "ebero_s", [11, 102], "Stroke subtype (ebero_s) when ebero_n==1"),
    ("emi_n",   "emi_s",   [41, 101], "MI subtype (emi_s) when emi_n==1"),
]:
    event_mask = df[ind_col] == 1
    counts = df.loc[event_mask, subtype_col].value_counts().sort_index()
    unknown = counts[~counts.index.isin(valid)]
    print(f"\n{title}")
    print(counts.rename("count").to_frame())
    if len(unknown) > 0:
        print(f"  ⚠ Unexpected subtypes: {unknown.index.tolist()}")
    else:
        print(f"  All subtypes in expected set {valid}")

## 7. Summary statistics for `e*_f` across all patients

In [ ]:
df[["edood_f", "ebero_f", "emi_f"]].describe()